# ⛰ Core Lab 3 — Train a Real Network
**A Billion Small Steps · Hardware for AI · Step 1 · Book sections 1.5, 1.C (read 1.6–1.7) · Foothold lessons 1.5, 1.C**

Read the book sections first, then run this notebook from top to bottom. Runs on the **CPU** on purpose: the default runtime is fine.

**What you should see:** layers collapsing without an activation; about 95% accuracy on MNIST; seconds per epoch on the CPU; validation loss rising when a model memorises.

| Part | What you do | Book |
|---|---|---|
| A | Build the B-grade network, let training find its cut-offs, and prove a layer is a matrix multiply | 1.5 |
| B | Probabilities, then train a network that reads handwritten digits on the CPU; then catch it memorising | 1.C |

⏱️ About 2 hours. 📘 Textbook: D2L Ch 4, §5.1–5.3 · Hwu, Kirk & El Hajj Ch 16. Course site: https://aishugithub.github.io/course_GPU/

💬 **Every line of code has a comment** (after the `#`, or after `//` inside CUDA C). Read the comment, then the line it explains.

## ▶️ How to run this notebook

1. **Save your own copy first:** menu **File → Save a copy in Drive**. Work in that copy — the original is read-only, and your copy is what you submit.
2. **Run cells in order, top to bottom.** Click a grey code cell and press **Shift + Enter** (or the ▶ button on its left). Wait for it to finish (a ✓ and the output appear) before running the next one.
3. **No GPU needed for this lab.** The default CPU runtime is fine — that is deliberate: you will compare against the GPU later.
4. **Lost or confused?** Menu **Runtime → Restart session and run all** starts everything again from a clean slate.
5. Cells marked **✏️ YOUR TURN** are exercises: change the code, run it again, and write your answer in the text cell below it (double-click a text cell to edit it).

## 0 · Setup

In [ ]:
# Which versions is this session running? The book's printed outputs used Colab's free Tesla T4
# with PyTorch 2.14.1 (CUDA 13.0 build) on 8 Nov 2026; small differences in your numbers are normal.
# If a cell fails after a Colab update: remove the "#" at the start of the next line, run this cell,
# then Runtime → Restart session (slow; only when needed).
# !pip install torch==2.14.1
import platform                                    # platform: tells us the Python version
import numpy, torch                                # NumPy and PyTorch, to read their version numbers
print("Python ", platform.python_version())        # the Python version
print("NumPy  ", numpy.__version__)                # the NumPy version
print("PyTorch", torch.__version__)                # the PyTorch version (and its CUDA build, after the +)
try:                                               # CuPy is only used by the CUDA C kernel labs
    import cupy                                    # try to load it
    print("CuPy   ", cupy.__version__)             # its version
except Exception:                                  # not installed, or no GPU for it in this session
    print("CuPy    not available in this session") # that is fine for this notebook
if torch.cuda.is_available():                      # is there an NVIDIA GPU in this session?
    print("GPU    ", torch.cuda.get_device_name(0))  # yes: its name (the book used a Tesla T4)
else:                                              # no GPU
    print("GPU     none (CPU runtime)")            # say so

In [ ]:
import numpy as np                 # NumPy: arrays of numbers and fast maths on them ("np" is its short name)
import matplotlib.pyplot as plt    # Matplotlib: draws charts and pictures ("plt" is its short name)
import time                        # time: we use its stopwatch, time.perf_counter()

# Print arrays neatly: wide lines, 3 decimal places, no 1e-05 style numbers
np.set_printoptions(linewidth=150, precision=3, suppress=True)
print("Ready! NumPy version:", np.__version__)   # show that everything loaded, and NumPy's version

## Part A · Why More Than One Neuron, and Why a Layer Is a Matrix Multiply  *(Section 1.5)*

### A1 · Who got a B?
In Nila's college a **B grade means marks from 60 to 75**. One neuron has one cut-off, so it can only say "higher than X" or "lower than X". A B has two edges, so we split the question:

| Neuron | Asks | Weighted sum |
|---|---|---|
| 1 | at least 60? (YES for higher marks) | `z1 = 1 × m − 60` |
| 2 | 75 or below? (YES for lower marks) | `z2 = −1 × m + 75` |
| 3 | both yes? | `z3 = 1 × a1 + 1 × a2 − 1.5` |

Neurons 1 and 2 read the **same** input (the marks `m`), each with its own weight and bias: together they are a **layer**. Neuron 3 reads their answers `a1`, `a2`.

*Same idea, different world:* a room feels comfortable between 20 °C and 28 °C. That question also has two edges, so it needs the same three neurons.

In [ ]:
def step(z):                               # the step activation
    return 1 if z >= 0 else 0              # 1 = YES when z is 0 or more, otherwise 0 = NO

def b_grade(m, activation=step):           # the 3-neuron B-grade network for marks m (step, unless told otherwise)
    z1 = 1 * m - 60                        # neuron 1, "at least 60?": weight +1, bias -60, so z1 >= 0 from 60 marks up
    z2 = -1 * m + 75                       # neuron 2, "75 or below?": weight -1, bias +75, so z2 >= 0 up to 75 marks
    a1, a2 = activation(z1), activation(z2)   # each neuron's answer, after its activation
    z3 = 1 * a1 + 1 * a2 - 1.5             # neuron 3, "both yes?": only a1 + a2 = 2 gets above 1.5
    return z1, a1, z2, a2, z3              # hand back every number so we can print the whole trace

print("marks |   z1  a1 |   z2  a2 |   z3  | answer")   # the table heading
for m in [52, 68, 81]:                     # three students: below the band, inside it, above it
    z1, a1, z2, a2, z3 = b_grade(m)        # run the network and unpack its five numbers
    # print one row: the two sums, their answers, z3, and B if z3 >= 0
    print(f"{m:5} | {z1:4} {a1:3} | {z2:4} {a2:3} | {z3:5} | {'B' if z3 >= 0 else 'NOT B'}")

Now **remove the activation**: each neuron passes on its raw sum `z` instead of a YES/NO answer.

In [ ]:
no_activation = lambda z: z                # "no activation": pass z straight through unchanged (lambda = a one-line function)

for m in [35, 52, 68, 81, 96]:             # five students, from low marks to high
    z1, a1, z2, a2, z3 = b_grade(m, activation=no_activation)   # the same network with the activation switched off
    print(f"marks {m}:  z3 = ({z1}) + ({z2}) - 1.5 = {z3}")      # show how z3 is built for this student

`z3` is **13.5 every time**: `(m − 60) + (75 − m) − 1.5 = 13.5`. The `+m` from neuron 1 and the `−m` from neuron 2 cancel, so the network gives a B to a student with 35 marks and to one with 96. Any mix of the two raw sums works out to `(a − b) × m + constant`: one weight and one bias, which is **just one neuron again**. The activation between the layers is what stops this collapse.

### A2 · Who picks the questions? Let training find them
Above, **we** chose the cut-offs 60 and 75. In a real network the designer only chooses **how many** neurons. Here: 2 hidden neurons (sigmoid) and 1 output neuron, starting from random weights, trained on 17 students' marks and grades. Gradient descent (Part D) does the rest. Afterwards we read each hidden neuron's cut-off, `−b ÷ w`.

In [ ]:
import torch                                   # PyTorch: the AI library we use for real networks
torch.manual_seed(0)                           # fix the "random" starting weights, so everyone gets the same numbers

# The marks of 17 students in Nila's class
m_data = torch.tensor([35., 41, 46, 50, 54, 58, 62, 64, 66, 68, 70, 72, 78, 82, 86, 91, 96])
y_data = ((m_data >= 60) & (m_data <= 75)).float()   # the label for each one: 1.0 = got a B, 0.0 = did not
# Shift and shrink the marks so the numbers stay near 0 (this helps training).
# unsqueeze(1) turns the list of 17 numbers into a column (17 rows × 1 input), the shape a layer expects.
x = ((m_data - 65) / 10).unsqueeze(1)

hidden = torch.nn.Linear(1, 2)                 # layer 1: 2 neurons, each with its own (random) weight and bias
output = torch.nn.Linear(2, 1)                 # layer 2: 1 neuron that combines their two answers
def network(x):                                # the whole network: input → layer 1 → sigmoid → layer 2
    return output(torch.sigmoid(hidden(x)))    # sigmoid between the layers: a smooth, learnable "YES/NO"

def cutoffs():                                 # read each hidden neuron's cut-off, in marks
    w = hidden.weight.detach().squeeze()       # the two hidden weights (detach = only read them, don't train them)
    b = hidden.bias.detach()                   # the two hidden biases
    return 65 + 10 * (-b / w)                  # cut-off = -b ÷ w, then undo the shift and shrink to get marks

loss_fn_bce = torch.nn.BCEWithLogitsLoss()     # the usual loss for YES/NO questions (binary cross-entropy)
# The optimizer does gradient descent (SGD) on all 4 weights and 3 biases, with learning rate 1.0
opt = torch.optim.SGD(list(hidden.parameters()) + list(output.parameters()), lr=1.0)
for step_no in range(1501):                    # 1,501 training steps
    loss = loss_fn_bce(network(x).squeeze(), y_data)   # predict all 17 students and score the answers
    opt.zero_grad()                            # wipe the slopes left over from the previous step
    loss.backward()                            # backpropagation: find the slope for every weight and bias
    opt.step()                                 # nudge every weight and bias a little against its slope
    if step_no in (0, 100, 300, 1500):         # report at only a few steps, to keep the output short
        # the fraction of the 17 students the network now gets right (output >= 0 means B)
        acc = ((network(x).squeeze() >= 0).float() == y_data).float().mean()
        c = cutoffs()                          # the two hidden neurons' cut-offs right now
        # print the step, the loss, the score and both cut-offs
        print(f"step {step_no:4d}: loss {loss.item():.3f}   correct {acc.item():.0%}   hidden cut-offs = {c[0]:6.1f} and {c[1]:6.1f} marks")

Nobody told the neurons "60" and "75". They started at meaningless cut-offs, and gradient descent moved them until they asked the useful questions. A neuron may ask its question the other way round (for example "above 75?") and let the output neuron subtract it; the cut-offs are what matter. **That is what "the machine learns" really means.**

### ✏️ YOUR TURN (A1–A2)
1. Change `torch.manual_seed(0)` to another number and run A2 again. Do the cut-offs still end up near 60 and 75? (Most seeds do; now and then training gets stuck, so try one more.)
2. Make a B mean 55 to 70 marks (change `y_data`). What cut-offs does training find?

*Your answers:*

### A3 · A layer is a matrix multiply
A layer of 128 neurons reading 784 inputs has a weight matrix `W` of shape (128, 784).
For a batch `X` of 64 images: `Y = X @ W.T + b`. We check that PyTorch's `nn.Linear` does **exactly** this.

In [ ]:
# Made-up data: a batch of 64 "images" of 784 numbers each, and random weights for 128 neurons
X = np.random.rand(64, 784).astype(np.float32)           # 64 rows (images) × 784 columns (pixels), random values 0..1
W = np.random.randn(128, 784).astype(np.float32) * 0.05  # one row of 784 weights for each of the 128 neurons
b = np.random.randn(128).astype(np.float32) * 0.05       # one bias for each neuron

Y_numpy = X @ W.T + b                      # the maths by hand: W.T is W turned on its side (transposed) so the shapes line up

layer = torch.nn.Linear(784, 128)          # PyTorch's ready-made layer: 784 inputs → 128 neurons...
with torch.no_grad():                      # (no_grad: we are not training here, so don't track slopes)
    layer.weight.copy_(torch.from_numpy(W))    # ...given OUR weights instead of its own random ones
    layer.bias.copy_(torch.from_numpy(b))      # ...and OUR biases
    Y_torch = layer(torch.from_numpy(X)).numpy()   # push the batch through the layer, then turn the answer back into NumPy

print("Output shape:", Y_numpy.shape)      # (64, 128): 128 outputs for each of the 64 images
# the biggest gap between the two answers: it should be tiny (only rounding)
print("Biggest difference between NumPy and PyTorch:", np.abs(Y_numpy - Y_torch).max())
print("MACs for this one layer, one batch:", f"{64 * 784 * 128:,}")   # images × inputs × neurons multiply-adds

The difference is essentially zero (tiny rounding only). **`nn.Linear` is a matrix multiply.** That is the operation GPUs are built to do fast.

### ✏️ YOUR TURN (A)
Time the plain-Python triple loop against `@` for a smaller case. Run it and note the ratio.

In [ ]:
n = 60                                                 # small matrices: 60 × 60
A = np.random.rand(n, n); B = np.random.rand(n, n)     # two random 60 × 60 matrices to multiply

t0 = time.perf_counter()                               # start the stopwatch
# Plain Python: work out every cell C[i][j] on its own, by adding up A[i,k] × B[k,j] for every k (one multiply-add each)
C = [[sum(A[i, k] * B[k, j] for k in range(n)) for j in range(n)] for i in range(n)]
t_loops = time.perf_counter() - t0                     # stop the stopwatch: the time the loops took

# The same product with NumPy's @, timed the same way (start, multiply, stop)
t0 = time.perf_counter(); C2 = A @ B; t_numpy = time.perf_counter() - t0

print(f"Python loops: {t_loops*1000:8.1f} ms")         # × 1000 turns seconds into milliseconds
# np.allclose checks that both answers are the same (apart from tiny rounding); also print how many times faster @ was
print(f"NumPy @     : {t_numpy*1000:8.3f} ms   → about {t_loops / t_numpy:,.0f}× faster (same answer: {np.allclose(C, C2)})")

*How many times faster was NumPy? (Core Lab 4 explains why.)*

## Part B · Your First Real AI — on the CPU  *(Section 1.C)*

A machine that reads the marks handwritten on answer sheets: a 784 → 128 → 10 network that reads MNIST handwritten digits, trained with PyTorch.
Everything you learned is here: tensors (Core Lab 1), neurons (Core Lab 2), layers (Part A), a loss and gradient descent (Core Lab 2).

We **time every epoch**. Keep this number: in Core Lab 9 a GPU trains MNIST networks, and in Core Lab 4 you will count this network's work.

### B0 · How sure is it? Answers as probabilities
Our pass/fail neuron never said PASS: it said a number like **0.94**, a **probability** (0 = never, 1 = certain; the probabilities of all the possible answers add up to 1).
Sigmoid gives the probability of YES for a two-way question. For **many** answers (a grade A, B, C or F; a digit 0–9; a chatbot's next word) the network has one output neuron per answer, and **softmax** turns their scores into probabilities:
**e to the power of each score** (all become positive), then **divide by the total** (they add up to 1).

In [ ]:
scores = np.array([2.0, 1.0, 0.1, -1.0])   # the 4 output neurons' scores for one student: A, B, C, F
e = np.exp(scores)   # softmax, move 1: e to the power of each score (all become positive)
probs = e / e.sum()   # softmax, move 2: divide by the total, so the four add up to 1
print(probs.round(3), "total:", probs.sum().round(3))   # the four probabilities, rounded, and their total
print("loss if the student really got a B:", round(-np.log(probs[1]), 3))   # cross-entropy: −log(probability of the right answer)

### ✏️ YOUR TURN (B0)
Change the score for F from −1.0 to 3.0 and re-run. Which grade wins now, and do the four probabilities still add up to 1?

*(How a chatbot uses these odds to pick its next word, and what the "temperature" setting does, is a Side Trail: Appendix A.3 and the Step 1 Side Trails notebook.)*

*Your answer:*

### B1 · Your first real AI: reading handwritten digits
The network below ends with **10 output neurons, one per digit**. `nn.CrossEntropyLoss` applies softmax to those 10 scores and takes −log of the right digit's probability, all in one step.

In [ ]:
import torch   # PyTorch: the toolkit for building and training AI models
from torch import nn                             # nn = PyTorch's building blocks for networks
from torchvision import datasets, transforms   # ready-made datasets (MNIST), and ways to convert images to tensors
from torch.utils.data import DataLoader          # DataLoader serves the data in batches

to_tensor = transforms.ToTensor()          # converts each picture to a tensor; pixels 0..255 → floats 0..1
train_data = datasets.MNIST("data", train=True,  download=True, transform=to_tensor)   # 60,000 to learn from
test_data  = datasets.MNIST("data", train=False, download=True, transform=to_tensor)   # 10,000 kept for testing
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)   # batches of 64, in a new random order each epoch
test_loader  = DataLoader(test_data,  batch_size=1000)               # big batches are fine for testing

model = nn.Sequential(         # the layers, applied one after another
    nn.Flatten(),              # 28×28 picture → one row of 784 numbers
    nn.Linear(784, 128),       # layer 1: 128 neurons, a (128 × 784) weight matrix
    nn.ReLU(),                 # the bend (activation)
    nn.Linear(128, 10),        # layer 2: 10 scores, one per digit 0–9; the biggest score is the guess
)
loss_fn = nn.CrossEntropyLoss()                            # the loss: how wrong are the 10 scores?
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)    # gradient descent on every weight, learning rate 0.1

print(model)   # show the network's layers
print("Parameters (knobs):", sum(p.numel() for p in model.parameters()))   # count every weight and bias

In [ ]:
device = "cpu"     # deliberately the CPU for now
model.to(device)   # put the model's weights on that device

for epoch in range(3):                         # 3 epochs = 3 full passes over the 60,000 images
    start = time.perf_counter()                # start the stopwatch for this epoch
    model.train()                              # switch the model to training mode
    for images, labels in train_loader:        # one batch of 64 images (and their labels) at a time
        images, labels = images.to(device), labels.to(device)   # move this batch to the device the model is on
        scores = model(images)                 # 1. predict
        loss = loss_fn(scores, labels)         # 2. measure the loss
        optimizer.zero_grad()                  #    (clear the previous batch's gradients)
        loss.backward()                        # 3. gradients (backpropagation)
        optimizer.step()                       # 4. nudge every weight
    seconds = time.perf_counter() - start      # stop the stopwatch

    # Check on the 10,000 test images the model has never trained on
    model.eval(); correct = 0                  # evaluation mode; reset the counter
    with torch.no_grad():                      # no training here, so no gradients needed
        for images, labels in test_loader:   # go through the test pictures, one batch at a time
            # argmax(1) = the digit with the highest score; count how many match the labels
            correct += (model(images.to(device)).argmax(1) == labels.to(device)).sum().item()
    print(f"epoch {epoch + 1}: loss {loss.item():.3f}   test accuracy {correct / len(test_data):.1%}   time {seconds:.1f} s")   # show this epoch's loss, test accuracy and time

Let's see the trained network read some digits it has **never seen**:

In [ ]:
images, labels = next(iter(test_loader))          # grab the first batch of test images
with torch.no_grad():   # just checking, not learning: keep no history
    guesses = model(images[:10]).argmax(1)        # the network's guess for the first 10

plt.figure(figsize=(10, 1.8))   # one wide, short drawing area
for i in range(10):   # the first 10 pictures
    plt.subplot(1, 10, i + 1)                     # 1 row, 10 boxes: draw in box i+1
    plt.imshow(images[i, 0], cmap="gray")         # images[i, 0] = picture i (0 = its only grey channel)
    plt.axis("off")   # no axes around the picture
    # write the guess above the picture: green if right, red if wrong
    plt.title(f"{guesses[i].item()}", color="green" if guesses[i] == labels[i] else "red")
plt.suptitle("The network's guesses (green = right, red = wrong)")   # a title over all the pictures
plt.show()   # draw it

How sure is the network about each of those digits? Softmax turns its 10 scores into 10 probabilities:

In [ ]:
with torch.no_grad():   # just checking, not learning: keep no history
    probs = torch.softmax(model(images[:10]), dim=1)   # softmax over the 10 scores of each picture (dim=1 = along each row)
for i in range(10):   # the same 10 pictures
    best = probs[i].argmax().item()   # the digit with the highest probability = the guess
    print(f"picture {i}: guess {best}, {probs[i, best].item():.1%} sure   (real digit {labels[i].item()})")   # the guess, how sure, and the truth

### ✏️ YOUR TURN (B) — record your results
| | Your value |
|---|---|
| Test accuracy after 3 epochs | |
| Seconds per epoch on the CPU | |
| Parameters in the model | |
| The lowest "% sure" among the 10 test pictures | |

Then: change `nn.Linear(784, 128)` and `nn.Linear(128, 10)` to use **512** hidden neurons and re-run both cells. What happens to the time per epoch?

Look at the "% sure" list. Is any guess **wrong yet very sure**? What does that tell you about a model that says it is "97% sure"?

*Your answers:*

### B2 · Catch the network memorising: training vs validation loss
Give the same 784 → 128 → 10 network only **200** training images, keep **10,000 other images** as a validation set it never trains on, and train for 100 epochs. Watch both losses. (About 1–2 minutes on Colab's CPU.)

In [ ]:
# the full 60,000 training images (Part B1 loaded them as train_data)
full_train = train_data
# a fixed random start, so everyone's run matches the book's
torch.manual_seed(0)
# a fresh copy of the Part B1 network: 784 → 128 → 10
model = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
# the same loss as Part B1
loss_fn = nn.CrossEntropyLoss()
# the same gradient descent, learning rate 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# only 200 training images: few enough to memorise
train_set = torch.utils.data.Subset(full_train, range(0, 200))
# 10,000 other images for validation: never trained on
val_set = torch.utils.data.Subset(full_train, range(50000, 60000))
# batches of 20 for training, shuffled every epoch
train_loader = DataLoader(train_set, batch_size=20, shuffle=True)
# big batches for checking (no training happens here)
val_loader = DataLoader(val_set, batch_size=1000)

# the average loss over a whole pile of images, without training
def average_loss(loader):
    # switch off the slope bookkeeping: we only measure
    with torch.no_grad():
        # loss of each batch × its size, added up, divided by the number of images
        return sum(loss_fn(model(x), y).item() * len(y) for x, y in loader) / len(loader.dataset)

# two lists to draw the two curves
train_hist, val_hist = [], []
# 100 epochs: far more than 200 images need
for epoch in range(1, 101):
    # the same four-step training loop as Part B1
    for images, labels in train_loader:
        # 1. predict and 2. measure the loss
        loss = loss_fn(model(images), labels)
        # wipe the old slopes
        optimizer.zero_grad()
        # 3. find the slopes
        loss.backward()
        # 4. nudge every weight
        optimizer.step()
    # after each epoch: score the training pile and the validation pile
    train_hist.append(average_loss(train_loader))
    # the validation loss is the honest one
    val_hist.append(average_loss(val_loader))
    # print a few epochs so the output stays short
    if epoch in (1, 5, 10, 20, 40, 60, 80, 100):
        # both losses side by side
        print(f"epoch {epoch:2}:  training loss {train_hist[-1]:.3f}   validation loss {val_hist[-1]:.3f}")

# the epoch with the lowest validation loss (counting from 1)
best = val_hist.index(min(val_hist)) + 1
# where to stop
print(f"lowest validation loss {min(val_hist):.3f} at epoch {best}")

In [ ]:
plt.figure(figsize=(6.5, 3.2))                                        # a new chart
plt.plot(range(1, 101), train_hist, label="training loss (200 images)")   # blue: the images it learns from
plt.plot(range(1, 101), val_hist, label="validation loss (10,000 images)") # orange: images it never trains on
plt.axvline(best, ls="--", color="green", label=f"stop here: epoch {best}") # the epoch with the lowest validation loss
plt.xlabel("epoch"); plt.ylabel("average loss")                       # axis labels
plt.legend(); plt.grid(alpha=0.3)                                     # colour key and faint grid
plt.title("Learning, then memorising")                                # the chart's title
plt.show()                                                            # draw it

### ✏️ YOUR TURN (B2)
| | Your value |
|---|---|
| Epoch with the lowest validation loss | |
| Lowest validation loss | |
| Validation loss at epoch 100 | |
| Training loss at epoch 100 | |

1. After the best epoch, the training loss keeps falling but the validation loss rises. In one sentence: what is the network doing?
2. Why must we never pick the best epoch (or the learning rate) using the **test** set?
3. Change `range(0, 200)` to `range(0, 2000)` (10× more training images) and re-run. Does the turning point come later?

*Your answers:*

---
## 🎓 Wrap-up
- A layer is a matrix multiply (`nn.Linear` = X @ W.T + b); without an activation between them, layers collapse into one.
- Softmax turns scores into probabilities; cross-entropy is −log of the right answer's probability.
- Your first real AI read handwritten digits at about 95%, and you timed it on the CPU.
- A validation set catches memorising: stop at the epoch with the lowest validation loss.
- **Next — Core Lab 4: How Much Work Is AI?** Count the multiply-adds, and measure what your CPU can do.